# Homework 1 – Clustering & Disease Phenotyping
**Dataset**: Tabula Muris Senis (single-cell RNA-seq across mouse organs, young & aged)

**Goal**: Identify tissue-specific phenotypes and cell-type clusters to understand single-cell transcriptomes.

## Questions
### Conceptual (answer in PDF)  (20 pts)
1. (3 points) Define a clinical phenotype. Why is unsupervised clustering useful for discovering disease subtypes?
2. (3 points) Compare k-means vs hierarchical clustering. What assumptions do they make about cluster shape/structure?
3. (4 points) How can dimensionality reduction (PCA) help in analyzing high-dimensional scRNA-seq data?
4. (5 points) In disease phenotyping using cluster analysis, why is it important to identify  clinically recognizable clusters? Put differently: what is/are the problems if the identified  clusters are not clinically meaningful?
5. (5 points) Partitioning Around Medoids (PAM) with Gower’s distance is designed for  clustering mixed data types. Why did K-means with Euclidean distance perform better on the clinical data (with mixed data types) in Lecture 5? Similarly, why did K-means with Euclidean distance outperform K-means with Manhattan distance on the same dataset?


### Coding & Analysis (80 pts)
Complete the TODO sections below and answer the Interpretation (written) questions in the PDF.

## 0. Setup & Data Loading
Load Tabula Muris Senis dataset and select 3 tissues (Limb_Muscle, Pancreas, Spleen) with ~5000 cells total.

In [ ]:
#if running in co-lab this chunk will download necessary packages
!pip install scanpy
!pip install scikit-misc
!pip install igraph leidenalg

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy import stats
from pathlib import Path
from statsmodels.stats.multitest import multipletests


# Create output directories
import os
os.makedirs('figs', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

In [5]:
# Load pre-subset AnnData object (~5000 cells, 3 tissues: Limb_Muscle, Pancreas, Spleen)

adata = sc.read_h5ad("HW1_2026_subset.h5ad")

# Print basic info
print(adata)
print("Tissues included:", adata.obs["tissue"].unique().tolist())
print("Number of cells:", adata.n_obs)
print("Number of genes:", adata.n_vars)

AnnData object with n_obs × n_vars = 5000 × 20138
    obs: 'age', 'cell', 'cell_ontology_class', 'cell_ontology_id', 'free_annotation', 'method', 'mouse.id', 'n_genes', 'sex', 'subtissue', 'tissue', 'tissue_free_annotation', 'tissue_selected'
    var: 'n_cells'
    uns: 'hw1_triplet_info'
    layers: 'counts', None (.X)
Tissues included: ['Spleen', 'Limb_Muscle', 'Pancreas']
Number of cells: 5000
Number of genes: 20138


## 1. Data Preprocessing (5 pts)
Filter, normalize, and log-transform.

In [7]:
# 1. Filter cells and genes
# (we give them the function calls, but they must fill in thresholds)
sc.pp.filter_cells(adata, min_genes=200)      # TODO: confirm threshold (e.g., 200)
sc.pp.filter_genes(adata, min_cells=3)     # TODO: confirm threshold (e.g., 3)

# 2. Normalize to 10,000 reads per cell
sc.pp.normalize_total(adata, target_sum=10000)  # TODO: confirm target sum from the comment above

# 3. Log-transform (natural log, add 1 inside log)
sc.pp.log1p(adata)

# Highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3")
adata_hvg = adata[:, adata.var["highly_variable"]].copy()

# Scale features
sc.pp.scale(adata_hvg, max_value=10)

print("Preprocessing complete.")
print(adata)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14316\3531476413.py:13: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3")
C:\Python313\Lib\functools.py:934: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


Preprocessing complete.
AnnData object with n_obs × n_vars = 5000 × 16557
    obs: 'age', 'cell', 'cell_ontology_class', 'cell_ontology_id', 'free_annotation', 'method', 'mouse.id', 'n_genes', 'sex', 'subtissue', 'tissue', 'tissue_free_annotation', 'tissue_selected'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hw1_triplet_info', 'log1p', 'hvg'
    layers: 'counts', None (.X)


## 2. Dimensionality Reduction (20 pts)
Apply PCA and visualize tissue separation. You may use any library of your choice (e.g., scanpy, pandas + sklearn).

In [ ]:
# TODO:
# 3. Apply PCA (50 components)
# 4. Plot first 2 PCs colored by tissue
N_PCs = 50 # Number of PCA components
# YOUR CODE HERE

# Save plot (WILL BE GRADED)
# save figure to ('figs/pca_by_tissue.png')


#### Interpretation (written) question:

*Answer in PDF:*

Do the tissues separate in PCA space? Explain your observation.


## 3. K-means Clustering (20 pts)
Determine optimal K and compare clusters to tissue labels.

In [ ]:
# TODO:
# import the tools you will use



N_PCS   = 50
K_RANGE = [2, 3, 4, 5, 6]
SEED    = 42
LABEL_COL = "tissue"

# TODO: ensure you are using exactly the first N_PCS PCs


# TODO: compute elbow (inertia) and silhouette over K_RANGE (use random_state=SEED, n_init=50)


# TODO: choose best_k by max silhouette; if tie, pick the smaller k


# TODO: fit final KMeans with best_k (use random_state=SEED, n_init=50)

# TODO: compute metrics (silhouette, ARI, NMI) using (X_pca_use, km_labels, y_true)
# km_sil = ...
# km_ari = ...
# km_nmi = ...
# print(f"KMeans  k={best_k}  Sil={km_sil:.3f}  ARI={km_ari:.3f}  NMI={km_nmi:.3f}")

# TODO: print or draw confusion table clusters × tissue
# optionally save to -> outputs/kmeans_confusion.csv


# TODO: plot and save k-selection curves (Elbow + Silhouette)
# save figure in -> figs/kmeans_kselection_silhouette.png
# save figure in -> figs/kmeans_kselection_elbow.png


# TODO: PCA scatter (PC1 vs PC2) colored by KMeans clusters →
# save figure in -> figs/pca_kmeans.png


# TODO: PCA scatter (PC1 vs PC2) colored by tissue
# save figure in -> figs/pca_tissue.png



#### Interpretation (written) question:

*Answer in PDF:*

How well do clusters align with tissue labels?

## 4. Alternative Clustering Method (20 pts)
Apply graph-based or hierarchical Clustering

In [ ]:
# Use Hierarchical or Graph-based (Leiden/Louvain) clustering as alternative clustering method

# TODO: Perform Louvain or Agglomerative (hierarchical) clustering on PCA-reduced data


# TODO: Save the resulting cluster labels

# TODO: Calculate silhouette score for k-means clusters (already computed earlier)

# TODO: Calculate silhouette score for your new clusters

# TODO: Print or display all silhouette scores for comparison (including k-means)

# TODO: Plot results (PCA) for new clusters (PC1 vs PC2)


#### Interpretation (written) question:

*Answer in PDF:*

How does your alternative clustering method compare to k-means clustering?

## 5. Marker Gene Identification (15 pts)
Find top 5 marker genes for one cluster using differential expression. In this part you just make the following code work with your data. You may variable names and structure while maintaining functionality.

In [ ]:
# Marker gene identification for each cluster

# Expression matrix: cells × genes (from preprocessed HVG data)
expression_matrix = adata_hvg.to_df()

# Choose cluster assignments (KMeans, Leiden, or Hierarchical)
cluster_assignments = km_labels   # <-- TODO: change here if analyzing Leiden/Hierarchical
y_true = adata_hvg.obs["tissue"].values

all_markers = {}

for clust in np.unique(cluster_assignments):
    # TODO: make a boolean mask for cells in this cluster
    mask = ...

    # TODO: find the majority tissue among cells in this cluster
    majority_tissue = ...

    # TODO: loop over all genes and compute Welch’s t-test (target vs other cells)
    scores, pvals = [], []
    for g in range(expression_matrix.shape[1]):
        expr_target = ...
        expr_other = ...
        t_stat, p_val = stats.ttest_ind(expr_target, expr_other, equal_var=False)
        scores.append(t_stat)
        pvals.append(p_val)

    # Multiple testing correction (already provided)
    _, pvals_adj, _, _ = multipletests(pvals, method="fdr_bh")

    # Collect results for this cluster
    markers = pd.DataFrame({
        "gene": expression_matrix.columns,
        "t_stat": scores,
        "pval": pvals,
        "pval_adj": pvals_adj,
        # TODO: compute log fold change (mean target − mean other)
        "logFC": ...
    })

    # TODO: filter markers by adjusted p-value < 0.05
    # HINT: use markers[markers["pval_adj"] < 0.05]
    # TODO: then rank by absolute logFC and select top 5
    top5 = ...

    # Add tissue label column for clarity
    top5.insert(0, "majority_tissue", majority_tissue)

    all_markers[clust] = top5
    print(f"\nTop 5 marker genes for cluster {clust} (majority tissue = {majority_tissue})")
    print(top5)

    # Save results per cluster
    top5.to_csv(f"outputs/top5_markers_cluster{clust}.csv", index=False)

print("\nMarker gene identification complete. Results saved in outputs/ folder.")


#### Interpretation (written) question:

*Answer in PDF:*

Are the identified markers similar for different tissues? Explain briefly why they are or are not similar.